# Modeleon — first model

Two primitives:

- **`Variable`** — a value or formula.
- **`MultiVariable`** — a concept made of Variables and sub-concepts. Becomes an Excel tab.

Run top-to-bottom. Open the resulting `.xlsx` files — every cell is a live formula.

In [1]:
# Local install — the notebook lives in packages/engine/notebooks/.
# For the published package: %pip install -q modeleon
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


## 1. Single-sheet P&L

In [2]:
import modeleon as mo

pnl = mo.MultiVariable("P&L")
pnl.revenue  = mo.Variable(1_000_000)
pnl.cogs_pct = mo.Variable(0.6)

pnl.cogs    = pnl.revenue * pnl.cogs_pct
pnl.profit  = pnl.revenue - pnl.cogs

pnl

MultiVariable(__floating__.m116236b40, components=4)

In [3]:
pnl.to_excel("pnl.xlsx")

## 2. Multi-sheet model with cross-sheet references

In [6]:
from datetime import date

forecast = mo.MultiVariable("Forecast")
forecast.assumptions = mo.MultiVariable("Assumptions") # Each sub-MultiVariable becomes its own Excel tab automatically 

with forecast.assumptions as assumptions:

    assumptions.n_periods   = mo.Variable(12, display_name="Periods")
    assumptions.start_date  = mo.Variable(date(2025, 1, 1))

    assumptions.timeline = mo.MultiVariable('Timeline')
    assumptions.timeline.periods = mo.recurrence(
            start=1,
            formula="{prev} + 1",
            periods=assumptions.n_periods,
            display_name="Period",
        )
    
    assumptions.timeline.months = mo.recurrence(
            start=forecast.assumptions.start_date,
            formula="EDATE({prev}, 1)",
            periods=forecast.assumptions.n_periods,
            display_name="Month",
        )
    
    assumptions.start_rev   = mo.Variable(1_000_000, unit="$", display_name="Starting Revenue")
    assumptions.growth      = mo.Variable(0.05,display_name="Monthly Growth %")
    assumptions.cogs_pct    = mo.Variable(0.6, display_name="COGS %")
    assumptions.tax_rate    = mo.Variable(0.25, display_name="Tax Rate")

forecast.pnl = mo.MultiVariable("P&L") # to not allow
with forecast.pnl as pnl:
    # Mount the Timeline MV onto P&L too so its cells appear there.
    # Sharing an MV across two parents triggers an internal clone so
    # both placements get their own cell addresses.
    pnl.timeline = assumptions.timeline

    # `recurrence` expresses period-over-period: period 0 = start,
    # periods 1+ apply the formula with `{prev}` bound to the previous
    # period's cell. Change the growth rate on Assumptions and every
    # later cell updates.
    pnl.revenue = mo.recurrence(
        start=forecast.assumptions.start_rev,
        formula="{prev} * (1 + {growth})",
        growth=forecast.assumptions.growth,
        display_name="Revenue",
        periods=assumptions.n_periods,
    )
    # Cross-sheet reference + `set_display_name` overrides the
    # auto-humanized "Cogs" label on a derived Variable.
    pnl.cogs  = (pnl.revenue * forecast.assumptions.cogs_pct).set_display_name("COGS")
    pnl.gross = pnl.revenue - pnl.cogs
    pnl.taxes = pnl.gross * forecast.assumptions.tax_rate
    pnl.net   = pnl.gross - pnl.taxes

forecast

MultiVariable(__floating__.m120490b30, components=2)

In [ ]:
forecast.to_excel("forecast.xlsx")